Import all the necessary libraries and initialize the Earth Engine API

In [8]:
import ee
import geemap
import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import shape
import os
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
print("Folder :", os.getcwd())
os.chdir("E:/SynologyDrive/Articles/Data_paper/08 - Soil depth/")

Folder : E:\SynologyDrive\Articles\Data_paper\08 - Soil depth


Idenfy the Google Earth Engine (GEE) API account. Carreful the valdiation code is to be insert in the bar at the top of the notebook if run in Visual code

In [14]:
ee.Authenticate()
ee.Initialize()

Select the area of interest (AOI) by loading a shapefile and converting it to a GeoDataFrame and selection the parameters for the data selection.

In [25]:
# Local file
raster_path = r".\data\Large_grid.tif"
with rasterio.open(raster_path) as src:
    image = src.read(1)
    mask = image > 0     
    results = (
        {'properties': {'value': v}, 'geometry': s}
        for i, (s, v) in enumerate(
            shapes(image, mask=mask, transform=src.transform))
    )

# Convert in GeoDataFrame
geoms = list(results)
gdf = gpd.GeoDataFrame.from_features(geoms)

# Create a polygon and gee object
aoi_poly = gdf.union_all()
aoi = ee.Geometry(mapping(aoi_poly))

Plot the AOI

In [26]:
Map = geemap.Map()
Map.centerObject(aoi, 8)
Map.addLayer(aoi, {}, "AOI")
Map

Map(center=[37.001263030021775, 42.75000000000002], controls=(WidgetControl(options=['position', 'transparent_…

Compute the Landsat 5 function and bands extraction

In [27]:
def maskL5clouds(image):
    qa = image.select('QA_PIXEL')

    cloudBit        = 1 << 3
    cloudShadowBit  = 1 << 4

    mask = (qa.bitwiseAnd(cloudBit).eq(0)
            .And(qa.bitwiseAnd(cloudShadowBit).eq(0)))
    return image.updateMask(mask)

dataset = (
    ee.ImageCollection("LANDSAT/LT05/C02/T1_TOA")
    .filterDate("1990-01-01", "2010-12-31")  # period
    .map(maskL5clouds)
)

# Median
L5_composite = dataset.median().clip(aoi)

Compute the Suomi National Polar-orbiting Partnership (Suomi NPP) function and bands extraction

In [28]:
dataset = (
    ee.ImageCollection("NOAA/VIIRS/001/VNP21A1D")
    .filterDate("2016-02-01", "2016-03-30")  # period
)

# Median
LST_01_composite = dataset.median()
LST_01 = LST_01_composite.select('LST_1KM').clip(aoi)

dataset = (
    ee.ImageCollection("NOAA/VIIRS/001/VNP21A1D")
    .filterDate("2016-04-01", "2016-05-30")  # period
)

# Median
LST_02_composite = dataset.median().clip(aoi)
LST_02 = LST_02_composite.select('LST_1KM').clip(aoi)

dataset = (
    ee.ImageCollection("NOAA/VIIRS/001/VNP21A1D")
    .filterDate("2016-06-01", "2016-07-30")  # period
)

# Median
LST_03_composite = dataset.median().clip(aoi)
LST_03 = LST_03_composite.select('LST_1KM').clip(aoi)

dataset = (
    ee.ImageCollection("NOAA/VIIRS/001/VNP21A1D")
    .filterDate("2016-10-01", "2016-11-30")  # period
)

# Median
LST_04_composite = dataset.median().clip(aoi)
LST_04 = LST_04_composite.select('LST_1KM').clip(aoi)

Export the LST as a local image and the Landsat as images to Google Drive, as they are to heavy to be exported locally.

In [ ]:
geemap.ee_export_image(
    LST_01,
    filename=r".\data\NPP\LST_feb_mar_raw.tif",
    scale=1000,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    LST_02,
    filename=r".\data\NPP\LST_apr_may_raw.tif",
    scale=1000,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    LST_03,
    filename=r".\data\NPP\LST_jun_jul_raw.tif",
    scale=1000,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    LST_04,
    filename=r".\data\NPP\LST_oct_nov_raw.tif",
    scale=1000,
    region=aoi,
    crs="EPSG:32638"
)

Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\08 - Soil depth\data\NPP\LST_feb_mar.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\08 - Soil depth\data\NPP\LST_apr_may.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\08 - Soil depth\data\NPP\LST_jun_jul.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\08 - Soil depth\data\NPP\LST_oct_nov.tif


In [30]:
L5_bands = ['B2', 'B3', 'B4', 'B7']

for band in L5_bands:
    task = ee.batch.Export.image.toDrive(
        image=L5_composite.select([band]),
        description = f"Landsat5_{band}_2010_MedianComposite",
        scale = 30,
        region = aoi,
        crs = 'EPSG:32638',
        folder = 'GoogleEarthEngine',
        maxPixels = 1e13
    )
    task.start()

Vizualize the different bands

In [31]:
Map = geemap.Map()

# Vizualization parameters
LST_vis = {'min': 273, 'max': 320, 'palette': ['blue', 'green', 'red']}

# Add layers
Map.centerObject(aoi, 9)
Map.addLayer(L5_composite.select(["B3", "B2", "B1"]), {"min": 0, "max": 0.3}, "Landsat 5 Median")
Map.addLayer(LST_01, LST_vis, 'Mean Surface Temperature for Feb-Mar 2016');
Map.addLayer(LST_02, LST_vis, 'Mean Surface Temperature for Apr-May 2016');
Map.addLayer(LST_03, LST_vis, 'Mean Surface Temperature for Jun-Jul 2016');
Map.addLayer(LST_04, LST_vis, 'Mean Surface Temperature for Oct-Nov 2016');

Map

Map(center=[37.001263030021775, 42.75000000000002], controls=(WidgetControl(options=['position', 'transparent_…

Image will be dowloaded localy from ther R code